In [1]:
# from google.colab import drive
# drive.mount('/content/drive')

In [2]:
# %%capture --no-stderr
# !pip install python-dotenv openAI langchain_core langchain_openai langchain langchain-community sentence_transformers rank_bm25

In [3]:
# <오픈AI 임베딩을 활용한 벡터들 간 코사인 유사도 계산>

# 라이브러리 불러오기
import os
from dotenv import load_dotenv
import numpy as np
from numpy import dot
from numpy.linalg import norm
import pandas as pd
from langchain_openai import OpenAIEmbeddings
from langchain_openai import OpenAI
# .env 파일에서 환경 변수 로드
load_dotenv("/content/.env")
# 환경 변수에서 API 키 가져오기
api_key = os.getenv("OPENAI_API_KEY")
api_key2 = os.getenv("HF_API_KEY")

In [4]:
embeddings = OpenAIEmbeddings(model="text-embedding-ada-002")
query_result = embeddings.embed_query('저는 배가 고파요')
print(query_result)

[-0.016708848997950554, -0.021796436980366707, 0.01521017961204052, -0.02723897434771061, -0.0367831327021122, 0.011857893317937851, -0.034574564546346664, -0.006744012236595154, -0.023912979289889336, -0.016840310767292976, -0.008111219853162766, 0.010871926322579384, -0.010622148402035236, -0.022598357871174812, 0.011240020394325256, -0.00493640685454011, 0.012087952345609665, -0.0028708064928650856, 0.00810464657843113, -0.01610412262380123, 0.001368028810247779, -0.014473991468548775, 0.018207518383860588, -0.012705824337899685, 0.003214251482859254, 0.006418643519282341, 0.00539652444422245, -0.019062023609876633, -0.009215502068400383, -0.0017237984575331211, 0.03449568897485733, -0.01651165634393692, 1.0655629921529908e-05, 0.0030959355644881725, 0.007375031244009733, -0.005560852121561766, -0.006428502965718508, 0.003424591151997447, -0.003828837303444743, -0.0022085653617978096, -0.022480040788650513, -0.010319785214960575, 0.011358336545526981, -0.024662313982844353, 0.013211

In [5]:
# <오픈AI 임베딩을 활용한 문장들 간 코사인 유사도 계산>

data = [
    '주식 시장이 급등했어요',
    '시장 물가가 올랐어요',
    '전통 시장에는 다양한 물품들을 팔아요',
    '부동산 시장이 점점 더 복잡해지고 있어요',
    '저는 빠른 비트를 좋아해요',
    '최근 비트코인 가격이 많이 변동했어요',
]
df = pd.DataFrame(data, columns=['text'])
df

,text
0,주식 시장이 급등했어요
1,시장 물가가 올랐어요
2,전통 시장에는 다양한 물품들을 팔아요
3,부동산 시장이 점점 더 복잡해지고 있어요
4,저는 빠른 비트를 좋아해요
5,최근 비트코인 가격이 많이 변동했어요


In [6]:
# 텍스트를 임베딩 벡터로 변환하는 함수 정의
def get_embedding(text):
  return embeddings.embed_query(text)

# DataFrame의 각 행에 대해 'text' 열의 내용을 임베딩 벡터로 변환
df['embedding'] = df.apply(lambda row: get_embedding(
        row.text,
    ), axis=1)

# 변환된 DataFrame 출력
df

,text,embedding
0,주식 시장이 급등했어요,"[-0.01370231993496418, -0.035045821219682693, ..."
1,시장 물가가 올랐어요,"[-0.00015702718519605696, -0.03539642691612243..."
2,전통 시장에는 다양한 물품들을 팔아요,"[-0.009272306226193905, -0.01461657416075468, ..."
3,부동산 시장이 점점 더 복잡해지고 있어요,"[-0.017441052943468094, -0.0031565295066684484..."
4,저는 빠른 비트를 좋아해요,"[-0.03796839341521263, -0.013386794365942478, ..."
5,최근 비트코인 가격이 많이 변동했어요,"[-0.01749446988105774, -0.021561449393630028, ..."


In [7]:
# 코사인 유사도 계산 함수
def cos_sim(A, B):
    return dot(A, B)/(norm(A)*norm(B))

# 주어진 쿼리와 가장 유사한 상위 3개의 문서를 반환하는 함수
def return_answer_candidate(df, query):
    # 쿼리 텍스트를 임베딩 벡터로 변환
    query_embedding = get_embedding(query)

    # DataFrame의 각 문서 임베딩과 쿼리 임베딩 간의 유사도 계산
    df["similarity"] = df.embedding.apply(lambda x: cos_sim(np.array(x), np.array(query_embedding)))

    # 유사도가 높은 순으로 정렬하고 상위 3개 문서 선택
    top_three_doc = df.sort_values("similarity", ascending=False).head(3)

    return top_three_doc

# 예시 쿼리로 유사한 문서 검색
sim_result = return_answer_candidate(df, '과일 값이 비싸다.')
sim_result

,text,embedding,similarity
2,전통 시장에는 다양한 물품들을 팔아요,"[-0.009272306226193905, -0.01461657416075468, ...",0.824548
1,시장 물가가 올랐어요,"[-0.00015702718519605696, -0.03539642691612243...",0.814571
0,주식 시장이 급등했어요,"[-0.01370231993496418, -0.035045821219682693, ...",0.805389


In [10]:
# <BGE-M3 임베딩을 활용한 문장들 간 코사인 유사도 계산>

# 라이브러리 불러오기
# from langchain.embeddings import HuggingFaceBgeEmbeddings
from langchain_huggingface import HuggingFaceEmbeddings

# BGE-M3 모델 초기화
embeddings = HuggingFaceEmbeddings(model_name='BAAI/bge-m3')

# 데이터셋 생성
data = [
    '주식 시장이 급등했어요',
    '시장 물가가 올랐어요',
    '전통 시장에는 다양한 물품들을 팔아요',
    '부동산 시장이 점점 더 복잡해지고 있어요',
    '저는 빠른 비트를 좋아해요',
    '최근 비트코인 가격이 많이 변동했어요',
]
df = pd.DataFrame(data, columns=['text'])

# DataFrame의 각 행을 벡터로 변환
df['embedding'] = df['text'].apply(get_embedding)
df

ValueError: Due to a serious vulnerability issue in `torch.load`, even with `weights_only=True`, we now require users to upgrade torch to at least v2.6 in order to use the function. This version restriction does not apply when loading files with safetensors.
See the vulnerability report here https://nvd.nist.gov/vuln/detail/CVE-2025-32434

In [ ]:
# 쿼리 예시로 유사 문서 검색
sim_result = return_answer_candidate(df, '과일 값이 비싸다')
sim_result